In [21]:
import re
from pathlib import Path
import pandas as pd
from PyPDF2 import PdfReader

VERIFY_FILE_RE = re.compile(
    r"^(?P<date>\d{4}-\d{2}-\d{2})_l(?P<level>\d+)_(?P<match_name>.+)_verify\.pdf$",
    re.I
)


# More permissive header regex:
# - DIV and CLASS can be empty
# - case-insensitive
SHOOTER_RE = re.compile(
    r'^(?P<bib>\d+)\s+(?P<name>.+?)\s+DIV:\s*(?P<div>[A-Z0-9]*)\s+'
    r'CLASS:\s*(?P<class>[A-Z0-9]*)\s+FACTOR:\s*(?P<pf>[A-Za-z]+)\s+'
    r'CATEGORY:\s*(?P<cat>.*)$',
    re.I
)

def _parse_shooter_line_fallback(line: str):
    """
    Fallback parser for slightly mangled headers.
    Expects:
      <bib> <name> DIV: ... CLASS: ... FACTOR: ... CATEGORY: ...
    but tolerates missing div/class/category values.
    """
    if not re.match(r'^\d+\s+', line):
        return None
    if "DIV:" not in line or "FACTOR:" not in line:
        return None

    try:
        # Split at DIV:
        left, right = line.split("DIV:", 1)
        left = left.strip()
        bib = left.split()[0]
        name = left[len(bib):].strip()

        # Now parse labeled sections in order
        div_part, rest = right.split("CLASS:", 1) if "CLASS:" in right else (right, "")
        div = div_part.strip()

        class_part, rest2 = rest.split("FACTOR:", 1) if "FACTOR:" in rest else ("", rest)
        class_ = class_part.strip()

        pf_part, cat_part = rest2.split("CATEGORY:", 1) if "CATEGORY:" in rest2 else (rest2, "")
        pf = pf_part.strip().split()[0] if pf_part.strip() else ""
        cat = cat_part.strip()

        return {
            "shooter": name,
            "div": div,
            "class": class_,
            "power_factor": pf,
            "cat": cat
        }
    except Exception:
        return None

def _parse_stage_line(line: str):
    toks = line.split()
    if len(toks) < 4 or toks[0] != "Stage":
        return None

    # We assume the canonical order is:
    # Stage N FACTOR PTS A C D Ded. MI NS PE OT Time
    # But we’ll be defensive and read from the front and end.

    stg = toks[1]
    factor = toks[2] if len(toks) > 2 else ""
    pts = toks[3] if len(toks) > 3 else ""
    a = toks[4] if len(toks) > 4 else ""
    c = toks[5] if len(toks) > 5 else ""
    d = toks[6] if len(toks) > 6 else ""

    # Whatever remains tends to include DED/MI/NS/PE/(OT)/Time.
    rest = toks[7:]

    # Time is almost always last token
    time = rest[-1] if rest else ""

    # Try to map the 4 penalty tokens if present
    ded = rest[0] if len(rest) > 0 else ""
    mi  = rest[1] if len(rest) > 1 else ""
    ns  = rest[2] if len(rest) > 2 else ""
    pe  = rest[3] if len(rest) > 3 else ""

    ot = ""  # OT is not reliably separated in extracted text

    return dict(
        stg=stg, factor=factor, pts=pts, a=a, c=c, d=d,
        ded=ded, mi=mi, ns=ns, pe=pe, ot=ot, time=time
    )

def _try_parse_shooter(line: str):
    m = SHOOTER_RE.match(line)
    if m:
        return {
            "shooter": m.group("name").strip(),
            "div": m.group("div").strip(),
            "class": m.group("class").strip(),
            "power_factor": m.group("pf").strip(),
            "cat": m.group("cat").strip(),
        }
    # Fallback label-based parse
    return _parse_shooter_line_fallback(line)

def parse_verify_pdf_text(pdf_path: str | Path) -> pd.DataFrame:
    reader = PdfReader(str(pdf_path))
    rows = []
    current = None
    header_buf = None

    for page in reader.pages:
        text = page.extract_text() or ""
        for raw_line in text.splitlines():
            line = raw_line.strip()
            if not line:
                continue

            # If previous line looked like a partial header, try combining
            if header_buf:
                combined = f"{header_buf} {line}".strip()
                shooter = _try_parse_shooter(combined)
                if shooter:
                    current = shooter
                    header_buf = None
                    continue
                # If still no success, keep going (and drop buffer)
                header_buf = None

            shooter = _try_parse_shooter(line)
            if shooter:
                current = shooter
                continue

            # Heuristic: header line that got wrapped before FACTOR/CATEGORY
            if re.match(r'^\d+\s+.+\s+DIV:\s*', line, flags=re.I) and "FACTOR:" not in line:
                header_buf = line
                continue

            if line.startswith("Stage") and current:
                st = _parse_stage_line(line)
                if st:
                    rows.append({**current, **st})

    return pd.DataFrame(rows)

def get_match_data_text(match_file: str | Path) -> pd.DataFrame:
    path = Path(match_file)
    m = VERIFY_FILE_RE.search(path.name)

    if m:
        match_date = m.group("date")
        match_level = m.group("level")
        raw_name = m.group("match_name")

        # Build a normalized composite name if you want
        match_name = f"{match_date}_L{match_level}_{raw_name}"
    else:
        # Fallback if filename doesn't follow the convention
        match_date = path.name[:10]
        match_level = ""
        # everything before .pdf
        match_name = path.stem.replace("_verify", "")

    df = parse_verify_pdf_text(path)
    if df.empty:
        return df

    df["match_name"] = raw_name
    df["match_date"] = match_date
    df["match_level"] = match_level

    for col in ["factor", "time"]:
        df[col] = (
            df[col].astype(str)
                  .str.replace(",", ".", regex=False)
                  .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["pts"] = pd.to_numeric(df["pts"], errors="coerce").fillna(0).astype(int)
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    # df["hit_factor"] = (df["pts"] / df["time"]).round(4)

    year = match_date[:4]
    df['match_name'] = df['match_name'] + f"_{year}"
    cols = [
        "match_name", "match_level", "match_date",
        "shooter", "div", "class", "power_factor", "cat",
        "stg", "factor", "time", "pts", "a", "c", "d",
        "ded", "mi", "ns", "pe", "ot",
    ]
    df = df[[c for c in cols if c in df.columns]]
    cols = [
        "match_name", "match_level", "match_date",
        "shooter_name", "shooter_div", "shooter_class", "shooter_pf", "shooter_cat",
        "stg_n", "hf", "time", "pts", "a", "c", "d",
        "ded", "mi", "ns", "pe", "ot",
    ]
    df.columns = cols
    
    cols = ['a','c','d','mi','ns', 'stg_n', 'match_level']
    df[cols] = df[cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
    
    
    return df


def collect_matches_text(pdf_dir: str | Path) -> pd.DataFrame:
    pdf_dir = Path(pdf_dir)
    files = sorted(pdf_dir.rglob("*_verify.pdf"))

    out = []
    for f in files:
        df = get_match_data_text(f)
        if not df.empty:
            out.append(df)

    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()


In [22]:
df = collect_matches_text("pdf")

In [23]:
df[df["shooter_name"].str.contains("BUZZELLI, ANTONIO", na=False)].groupby(['match_name', 'match_date']).agg({'stg_n': 'size'})
# [
#     ["match_name", "shooter", "div", "class", "stg", "pts", "time", "factor"]
# ]

,,stg_n
match_name,match_date,
beretta_2025,2025-05-30,14
christmas_2025,2025-12-14,12
ita_4_2025,2025-07-06,10
ma5_1_2026,2026-02-15,8
ma5_2_2025,2025-02-23,8
ma5_2_2026,2026-03-01,8
ma5_3_2025,2025-03-08,8
ma5_4_2025,2025-03-23,8
ma5_5_2025,2025-04-13,8


In [24]:

df['stg_max_pts'] = (5 * (df['a'] + df['c'] + df['d'] + df['mi'])).astype(int)
df['pts_pct'] = (df['pts'] / df['stg_max_pts']).round(4)
df['winner_time'] = df.groupby(['match_name','match_date','shooter_div','stg_n'])['time'] \
    .transform(lambda s: s.loc[df.loc[s.index, 'hf'].idxmax()])
df['pct_winner_time'] = df['winner_time'] / df['time']
df['stg_max_hf'] = df.groupby(
    ['match_name', 'match_date', 'shooter_div', 'stg_n']
)['hf'].transform('max')
df['hf_pct'] = (df['hf'] / df['stg_max_hf']).round(4)
df['stg_match_pts'] = df['stg_max_pts'] * df['hf_pct']
df['hf'] = (df['pts'] / df['time']).round(4)
df[(df['match_name'] == 'monster_2025') & (df['shooter_name'] == 'PERAZZOLI, FRANCESCO')]


,match_name,match_level,match_date,shooter_name,shooter_div,shooter_class,shooter_pf,shooter_cat,stg_n,hf,...,ns,pe,ot,stg_max_pts,pts_pct,winner_time,pct_winner_time,stg_max_hf,hf_pct,stg_match_pts
17832,monster_2025,3,2025-08-03,"PERAZZOLI, FRANCESCO",P,G,Minor,,1,7.7975,...,0,0,,160,0.9625,19.75,1.000000,7.7975,1.0000,160.000
17833,monster_2025,3,2025-08-03,"PERAZZOLI, FRANCESCO",P,G,Minor,,2,6.8702,...,0,0,,60,0.9000,8.11,1.031807,7.1517,0.9606,57.636
17834,monster_2025,3,2025-08-03,"PERAZZOLI, FRANCESCO",P,G,Minor,,3,7.5899,...,0,0,,120,0.9500,13.38,0.890812,7.9223,0.9580,114.960
17835,monster_2025,3,2025-08-03,"PERAZZOLI, FRANCESCO",P,G,Minor,,4,7.2993,...,0,0,,60,1.0000,7.58,0.922141,7.6517,0.9539,57.234
17836,monster_2025,3,2025-08-03,"PERAZZOLI, FRANCESCO",P,G,Minor,,5,6.0733,...,0,0,,60,0.9667,7.46,0.781152,6.7024,0.9061,54.366
17837,monster_2025,3,2025-08-03,"PERAZZOLI, FRANCESCO",P,G,Minor,,6,8.0877,...,0,0,,120,0.9833,14.13,0.968472,8.2095,0.9852,118.224
17838,monster_2025,3,2025-08-03,"PERAZZOLI, FRANCESCO",P,G,Minor,,7,5.3416,...,0,0,,160,0.8062,23.02,0.953209,6.6030,0.8090,129.440
17839,monster_2025,3,2025-08-03,"PERAZZOLI, FRANCESCO",P,G,Minor,,8,5.6509,...,0,0,,60,0.9333,7.68,0.774975,7.5521,0.7483,44.898
17840,monster_2025,3,2025-08-03,"PERAZZOLI, FRANCESCO",P,G,Minor,,9,5.5675,...,0,0,,60,0.8667,8.38,0.897216,6.9212,0.8044,48.264
17841,monster_2025,3,2025-08-03,"PERAZZOLI, FRANCESCO",P,G,Minor,,10,8.4746,...,0,0,,120,1.0000,13.33,0.941384,8.8522,0.9573,114.876


In [26]:
# Check for matches where shooters have different stage counts than the median (indicating potential parsing issues)
stg_shooter = df.groupby(['match_name', 'match_date', 'shooter_name'])['stg_n'].count().reset_index()
match_stg_median = stg_shooter.groupby(['match_name', 'match_date'])['stg_n'].median()
for match, median in match_stg_median.items():
    match_df = df[df['match_name'] == match]
    shooters = match_df['shooter_name'].unique()
    for shooter in shooters:
        shooter_df = match_df[match_df['shooter_name'] == shooter]
        count_stg = shooter_df.shape[0]
        if count_stg != median:
            print(match, shooter)

In [25]:
df.to_csv("IPSC.csv", index=False)

In [27]:
def stage_standing(df, match, shooter_div, stg_n):
    stage_df = df[
        (df['match_name'] == match) &
        (df['shooter_div'] == shooter_div) &
        (df['stg_n'] == stg_n)
    ].copy().sort_values('hf', ascending=False)
    stage_df['rank'] = stage_df['hf'].rank(method='max', ascending=False)
    return stage_df[['rank', 'shooter_name', 'shooter_class', 'pts', 'time', 'hf', 'stg_match_pts', 'hf_pct']]

stage_standing(df, 'ma5_1_2026', 'P', 6)

,rank,shooter_name,shooter_class,pts,time,hf,stg_match_pts,hf_pct
32621,1.0,"PASSALIA, GIANLUCA",G,152,18.55,8.1941,160.000,1.0000
32085,2.0,"PANITTI, ANGELO",M,156,20.76,7.5145,146.736,0.9171
31773,3.0,"NARDIELLO, FRANCESCO",A,150,19.97,7.5113,146.672,0.9167
31685,4.0,"PETTINELLI, ALESSANDRO",G,140,18.65,7.5067,146.576,0.9161
32069,5.0,"CASSANELLI, PAOLO GIOVANNI",A,146,20.04,7.2854,142.256,0.8891
...,...,...,...,...,...,...,...,...
32285,113.0,"LOMBARDO, SEBASTIANO",C,120,45.23,2.6531,51.808,0.3238
31117,114.0,"FOGLIANI, EMANUELE",C,113,47.89,2.3596,46.080,0.2880
30981,115.0,"PAPAGNO, ANDREA",D,75,34.99,2.1435,41.856,0.2616
32333,116.0,"NEAGU MOISA, MIHAELA ALINA",D,125,60.38,2.0702,40.416,0.2526


In [28]:
def match_standing(data, match, shooter_div):
    stages = data['stg_n'].unique()
    stages_concat = []
    for stage in stages:
        stages_concat.append(stage_standing(data, match, shooter_div, stage)[['shooter_name', 'shooter_class', 'stg_match_pts']])
    standing = pd.concat(stages_concat).groupby(['shooter_name', 'shooter_class'])['stg_match_pts'].sum().reset_index().sort_values('stg_match_pts', ascending=False)
    standing['rank'] = standing['stg_match_pts'].rank(method='max', ascending=False)
    standing['pct'] = (standing['stg_match_pts'] / standing['stg_match_pts'].max()).round(4)
    standing['shooter_div'] = shooter_div
    return standing[['shooter_div', 'rank', 'shooter_name', 'shooter_class', 'stg_match_pts', 'pct']]
match_standing(df, 'monster', 'P')

,shooter_div,rank,shooter_name,shooter_class,stg_match_pts,pct


In [29]:
def override_stage_results(
    data, match, match_date, shooter_name, stg_n,
    time=None, a=None, c=None, d=None, mi=None, ns=None
):
    # Work on a copy so original df is not changed
    out = data.copy(deep=True)

    mask = (
        (out['match_name'] == match) &
        (out['match_date'] == match_date) &
        (out['shooter_name'] == shooter_name) &
        (out['stg_n'] == stg_n)
    )

    idx = out.index[mask]
    if idx.empty:
        print("No matching record found to override.")
        return out

    # If you truly expect exactly one, take the first
    idx = idx[0]

    # Override fields (only when provided)
    if a is not None: out.at[idx, 'a'] = a
    if c is not None: out.at[idx, 'c'] = c
    if d is not None: out.at[idx, 'd'] = d
    if mi is not None: out.at[idx, 'mi'] = mi
    if ns is not None: out.at[idx, 'ns'] = ns
    if time is not None: out.at[idx, 'time'] = time

    # Recompute dependent fields
    out.at[idx, 'pts'] = (
        5 * out.at[idx, 'a'] +
        3 * out.at[idx, 'c'] +
        1 * out.at[idx, 'd'] -
        10 * out.at[idx, 'mi'] -
        10 * out.at[idx, 'ns']
    )

    out.at[idx, 'pts_pct'] = round(out.at[idx, 'pts'] / out.at[idx, 'stg_max_pts'], 4) if out.at[idx, 'stg_max_pts'] else 0

    t = out.at[idx, 'time']
    out.at[idx, 'hf'] = round(out.at[idx, 'pts'] / t, 4) if t and t > 0 else 0

    max_hf = out.at[idx, 'stg_max_hf']
    out.at[idx, 'hf_pct'] = round(out.at[idx, 'hf'] / max_hf, 4) if max_hf and max_hf > 0 else 0

    out.at[idx, 'stg_match_pts'] = out.at[idx, 'stg_max_pts'] * out.at[idx, 'hf_pct']

    return out

In [30]:
df[(df['match_name'] == 'monster') & (df['shooter_name'] == 'BUZZELLI, ANTONIO') & (df['stg_n'] == 1)]

,match_name,match_level,match_date,shooter_name,shooter_div,shooter_class,shooter_pf,shooter_cat,stg_n,hf,...,ns,pe,ot,stg_max_pts,pts_pct,winner_time,pct_winner_time,stg_max_hf,hf_pct,stg_match_pts


In [31]:
od = override_stage_results(df, 'monster', '2025-08-03', 'BUZZELLI, ANTONIO', 1, time=20)
od[(od['match_name'] == 'monster') & (od['shooter_name'] == 'BUZZELLI, ANTONIO') & (od['stg_n'] == 1)]

No matching record found to override.


,match_name,match_level,match_date,shooter_name,shooter_div,shooter_class,shooter_pf,shooter_cat,stg_n,hf,...,ns,pe,ot,stg_max_pts,pts_pct,winner_time,pct_winner_time,stg_max_hf,hf_pct,stg_match_pts
